In [1]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="javyduck/SafeAuto-BDDX",
    repo_type="dataset",
    local_dir="data/SafeAuto-BDDX",
    allow_patterns=["BDDX_Processed.tar", "BDDX_Test.tar"],
)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

BDDX_Processed.tar:   0%|          | 0.00/6.17G [00:00<?, ?B/s]

BDDX_Test.tar:   0%|          | 0.00/781M [00:00<?, ?B/s]

'/workspace/VLM_experiments/data/SafeAuto-BDDX'

In [ ]:
import json
import os
from pathlib import Path
import matplotlib.pyplot as plt

# Set up paths
data_dir = Path("data/SafeAuto-BDDX")
sample_annotation = data_dir / "BDDX_Processed" / "info" / "02d478d1-e6811391_03238.json"
sample_video = data_dir / "BDDX_Processed" / "videos" / "0124dfa6-30a430dc_02950.mp4"

# Load annotation data
with open(sample_annotation, 'r') as f:
    annotation = json.load(f)

print("=== SAMPLE ANNOTATION DATA ===")
print(f"Sample ID: {annotation['id']}")
print(f"Video path: {annotation['video']}")
print(f"Video file exists: {sample_video.exists()}")
print(f"Video file size: {sample_video.stat().st_size / (1024*1024):.2f} MB")

print("\n=== ACTION & JUSTIFICATION ===")
for comment in annotation['comment']:
    if 'action' in comment:
        print(f"Action: {comment['action']}")
    if 'justification' in comment:
        print(f"Justification: {comment['justification']}")

print("\n=== CAR SENSOR DATA ===")
car_info = annotation['car_info']
print(f"Number of data points: {len(car_info['speed'])}")
print(f"Speed range: {min(car_info['speed']):.1f} - {max(car_info['speed']):.1f} units")
print(f"Acceleration range: {min(car_info['acceleration']):.3f} - {max(car_info['acceleration']):.3f}")
print(f"Course range: {min(car_info['course']):.3f} - {max(car_info['course']):.3f}")
print(f"Curvature range: {min(car_info['curvature']):.4f} - {max(car_info['curvature']):.4f}")

# Plot sensor data
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Car Sensor Data Over Time')

# Speed
ax1.plot(car_info['speed'], 'b-', linewidth=2)
ax1.set_title('Speed')
ax1.set_ylabel('Speed (units)')
ax1.grid(True, alpha=0.3)

# Acceleration
ax2.plot(car_info['acceleration'], 'r-', linewidth=2)
ax2.set_title('Acceleration')
ax2.set_ylabel('Acceleration')
ax2.grid(True, alpha=0.3)

# Course
ax3.plot(car_info['course'], 'g-', linewidth=2)
ax3.set_title('Course')
ax3.set_ylabel('Course (radians?)')
ax3.set_xlabel('Time step')
ax3.grid(True, alpha=0.3)

# Curvature
ax4.plot(car_info['curvature'], 'm-', linewidth=2)
ax4.set_title('Curvature')
ax4.set_ylabel('Curvature')
ax4.set_xlabel('Time step')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== DATASET STRUCTURE ===")
print("Available video files:")
video_dir = data_dir / "BDDX_Processed" / "videos"
videos = list(video_dir.glob("*.mp4"))
print(f"Number of videos in processed set: {len(videos)}")
for video in videos[:5]:  # Show first 5
    print(f"  {video.name}")
if len(videos) > 5:
    print(f"  ... and {len(videos) - 5} more")

print("\nAvailable annotation files:")
info_dir = data_dir / "BDDX_Processed" / "info"
annotations = list(info_dir.glob("*.json"))
print(f"Number of annotations in processed set: {len(annotations)}")
for ann in annotations[:5]:  # Show first 5
    print(f"  {ann.name}")
if len(annotations) > 5:
    print(f"  ... and {len(annotations) - 5} more")

In [ ]:
import cv2
from PIL import Image
import numpy as np

def extract_video_frames(video_path, num_frames=5, max_width=640):
    """Extract frames from video for inspection"""
    cap = cv2.VideoCapture(str(video_path))
    
    if not cap.isOpened():
        print(f"Error opening video file: {video_path}")
        return []
    
    # Get video properties
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    print(f"Video properties:")
    print(f"  Total frames: {total_frames}")
    print(f"  FPS: {fps:.1f}")
    print(f"  Resolution: {width}x{height}")
    print(f"  Duration: {total_frames/fps:.1f} seconds")
    
    frames = []
    frame_indices = np.linspace(0, total_frames-1, num_frames, dtype=int)
    
    for frame_idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if ret:
            # Convert BGR to RGB
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            
            # Resize if too wide
            if width > max_width:
                aspect_ratio = height / width
                new_width = max_width
                new_height = int(new_width * aspect_ratio)
                frame_rgb = cv2.resize(frame_rgb, (new_width, new_height))
            
            frames.append(frame_rgb)
    
    cap.release()
    return frames

# Extract and display frames from the sample video
print("=== VIDEO FRAME EXTRACTION ===")
frames = extract_video_frames(sample_video, num_frames=5)

if frames:
    # Display frames
    fig, axes = plt.subplots(1, len(frames), figsize=(15, 4))
    if len(frames) == 1:
        axes = [axes]
    
    for i, frame in enumerate(frames):
        axes[i].imshow(frame)
        axes[i].set_title(f'Frame {i+1}')
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("No frames could be extracted from the video.")

# Show another sample annotation
print("\n=== ANOTHER SAMPLE ANNOTATION ===")
# Let's find another annotation file
other_annotations = list(info_dir.glob("*.json"))
if len(other_annotations) > 1:
    other_ann_path = other_annotations[1]  # Second annotation
    with open(other_ann_path, 'r') as f:
        other_ann = json.load(f)
    
    print(f"Sample ID: {other_ann['id']}")
    for comment in other_ann['comment']:
        if 'action' in comment:
            print(f"Action: {comment['action']}")
        if 'justification' in comment:
            print(f"Justification: {comment['justification']}")
    
    # Show speed range for comparison
    speed_range = f"{min(other_ann['car_info']['speed']):.1f} - {max(other_ann['car_info']['speed']):.1f}"
    print(f"Speed range: {speed_range} units")